<p>
<font size='5' face='Georgia, Arial'>IIC-2233 Apunte Programación Avanzada</font><br>
<font size='1'>&copy; 2015 Karim Pichara - Christian Pieringer. Todos los derechos reservados. Editado por Equipo Docente IIC2233 2018-1 al 2026-2</font>
</p>

# Tabla de contenidos

1. [Ejecución de los ejemplos](#Ejecución-de-los-ejemplos)
2. [*Networking* e interfaces gráficas](#Networking-e-interfaces-gráficas)
    1. [El protocolo](#El-protocolo)
3. [Ejemplo simple (`1_ejemplo_simple`)](#Ejemplo-simple-(1_ejemplo_simple))
    1. [El *thread* que escucha](#El-thread-que-escucha)
    2. [La lógica del cliente](#La-lógica-del-cliente)
    3. [Conectando todo en `main.py`](#Conectando-todo-en-main.py)
    4. [¿Qué pasa si el servidor se cae?](#¿Qué-pasa-si-el-servidor-se-cae?)
4. [Ejemplo avanzado (`2_ejemplo_avanzado`)](#Ejemplo-avanzado-(2_ejemplo_avanzado))
    1. [El servidor](#El-servidor)
    2. [El cliente: un *thread* para escuchar y un `QTimer` para el reloj](#El-cliente:-un-thread-para-escuchar-y-un-QTimer-para-el-reloj)
    3. [Acotación: ¿y con `threading`?](#Acotación:-¿y-con-threading?)

## Ejecución de los ejemplos

Jupyter no fue creado para ejecutar interfaces gráficas ni programas de red, por lo que en este _notebook_ el código se muestra como texto y **no** se ejecuta. Los ejemplos están en `scripts/4-gui-y-networking/`.

**Importante:** cada ejemplo tiene **dos programas distintos**, un servidor y un cliente, que se ejecutan por separado. Necesitas al menos dos terminales: una posicionada en la carpeta del servidor y otra en la carpeta del cliente. Ambos ejemplos usan el mismo puerto (4444), así que antes de iniciar el servidor de un ejemplo, detén el del otro con `Ctrl+C`. Siempre inicia primero el servidor:

```bash
# Terminal 1
cd scripts/4-gui-y-networking/1_ejemplo_simple/servidor
python3 main.py
```

```bash
# Terminal 2 (y 3, 4... si quieres más clientes)
cd scripts/4-gui-y-networking/1_ejemplo_simple/cliente
python3 main.py
```

## *Networking* e interfaces gráficas

En semanas anteriores vimos cómo programar con una arquitectura **cliente-servidor**, y la semana pasada, cómo separar un programa con interfaz en *front-end* y *back-end*. Ahora queremos juntar ambas cosas.

Normalmente, **el servidor no necesita interfaz gráfica**: su rol es procesar las solicitudes de los clientes, y nadie interactúa con él directamente. Por lo tanto, solo el cliente tiene interfaz:

![Servidor y cliente con front-end y back-end](img/networking-interfaces.png)

Si miramos el cliente en detalle, queda dividido así:

- ***Front-end***: la ventana. No sabe nada de *sockets*: solo muestra información y emite señales cuando el usuario hace algo.
- ***Back-end***: la lógica del cliente. Envía mensajes al servidor y procesa los que llegan.
- **Un _thread_ que escucha**: `recv` es **bloqueante**, así que si lo llamamos en el *thread* principal, la interfaz se congela hasta que llegue un mensaje. Por eso, como vimos en el primer *notebook*, la recepción va en otro *thread*, que avisa con una señal cada vez que llega algo.

Enviar, en cambio, es rápido (para mensajes pequeños), así que se puede hacer directamente desde el *thread* principal.

![Flujo de un cliente con interfaz gráfica](img/networking-gui.svg)

### El protocolo

Para que los mensajes no lleguen cortados ni pegados, usamos el mismo protocolo de la semana de *networking*: cada mensaje se envía precedido de **4 _bytes_ que indican su largo**, y se recibe por *chunks* hasta completar ese largo. Lo dejamos en un módulo `protocolo.py`, que es **idéntico en el cliente y en el servidor**, ya que ambos programas deben hablar el mismo protocolo.

```python
"""
Protocolo de comunicación (el mismo de la semana 11): cada mensaje se envía
precedido por 4 bytes que indican su largo, y se recibe por chunks hasta
completar ese largo.

Este archivo es idéntico en el cliente y en el servidor: ambos programas
deben hablar el mismo protocolo.
"""
import socket

TAMAÑO_CHUNK = 4096


def enviar(sock: socket.socket, mensaje: bytes) -> None:
    largo = len(mensaje).to_bytes(4, byteorder="big")
    sock.sendall(largo + mensaje)


def recibir(sock: socket.socket) -> bytes:
    largo = int.from_bytes(recibir_bytes(sock, 4), byteorder="big")
    return recibir_bytes(sock, largo)


def recibir_bytes(sock: socket.socket, cantidad: int) -> bytes:
    datos = bytearray()
    while len(datos) < cantidad:
        chunk = sock.recv(min(TAMAÑO_CHUNK, cantidad - len(datos)))
        # recv retorna b"" cuando la otra parte cerró la conexión
        if not chunk:
            raise ConnectionError("La conexión se cerró")
        datos.extend(chunk)
    return bytes(datos)
```

Fíjate en que `recibir_bytes` lanza `ConnectionError` cuando `recv` retorna `b""`, lo que ocurre cuando la otra parte cerró la conexión. Así, quien llame a `recibir` puede detectar una desconexión con un `try/except`.

## Ejemplo simple (`1_ejemplo_simple`)

Este programa levanta una ventana desde la cual se le puede pedir al servidor un saludo o un color. El servidor responde con uno al azar, y la respuesta se muestra en la ventana. La estructura de archivos es:

```
1_ejemplo_simple/
├── servidor/
│   ├── main.py
│   ├── constantes.py
│   └── protocolo.py
└── cliente/
    ├── main.py
    ├── constantes.py
    ├── protocolo.py
    ├── backend/
    │   └── logica.py
    └── frontend/
        └── ventana.py
```

El servidor es igual a los que vimos en la semana de *networking*: un *thread* por cliente, que recibe un comando y responde. Este código se encuentra en el archivo `1_ejemplo_simple/servidor/main.py`.

```python
import socket
from random import choice
from threading import Thread
from constantes import HOST_DEFECTO, PORT_DEFECTO, SALUDOS, COLORES
from protocolo import enviar, recibir


# Función encargada de escuchar a 1 cliente
def escuchar_cliente(socket_cliente: socket.socket, direccion: tuple) -> None:
    try:
        while True:
            mensaje = recibir(socket_cliente).decode("utf-8")
            print(f"[{direccion}] pidió: {mensaje}")
            if mensaje == "palabra":
                respuesta = choice(SALUDOS)
            else:  # mensaje == "color"
                respuesta = choice(COLORES)
            enviar(socket_cliente, respuesta.encode("utf-8"))

    except ConnectionError:
        # El cliente se desconectó: se termina el thread que lo atendía
        print(f"[{direccion}] se desconectó")
    finally:
        socket_cliente.close()
```

### El *thread* que escucha

En el cliente, el *back-end* está en `backend/logica.py`. Primero definimos un `QThread` cuya única tarea es escuchar al servidor: en un ciclo infinito llama a `recibir` (que bloquea hasta que llegue un mensaje) y emite una señal con cada mensaje. Si la conexión se pierde, emite otra señal y termina.

```python
import socket
from PyQt6.QtCore import QObject, QThread, pyqtSignal
from protocolo import enviar, recibir


# Primero definimos un QThread que recibe el socket y se dedica
# exclusivamente a escuchar al servidor. recv es bloqueante, por eso
# no puede ejecutarse en el thread principal: congelaría la interfaz
class EscucharQThread(QThread):
    # Señal que entrega cada mensaje recibido al componente lógico
    senal_mensaje = pyqtSignal(str)
    # Señal que avisa que se perdió la conexión
    senal_desconectado = pyqtSignal()

    def __init__(self, socket_cliente: socket.socket) -> None:
        super().__init__()
        self.socket_cliente = socket_cliente

    def run(self) -> None:
        print("Levantando el thread que escucha al servidor...")
        try:
            while True:
                mensaje = recibir(self.socket_cliente).decode("utf-8")
                # Emitimos la señal con el mensaje decodificado
                self.senal_mensaje.emit(mensaje)
        except (ConnectionError, OSError):
            # El servidor se cayó, o nosotros cerramos el socket
            print("Dejé de escuchar al servidor")
            self.senal_desconectado.emit()
```

### La lógica del cliente

Luego, la clase `Logica` crea el *socket*, se conecta al servidor, inicia el *thread* que escucha (guardando su referencia) y procesa los mensajes que llegan. `procesar_mensaje` está conectado a la señal del *thread*, así que se ejecuta en el *thread* principal y puede emitir señales al *front-end* sin problemas.

```python
class Logica(QObject):

    senal_actualizar_etiqueta = pyqtSignal(str)
    senal_desconectado = pyqtSignal()

    def __init__(self, host: str, port: int) -> None:
        super().__init__()
        self.host = host
        self.port = port
        self.socket_cliente = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
        self.thread_escuchar = None

    def conectar_a_servidor(self) -> bool:
        try:
            self.socket_cliente.connect((self.host, self.port))
        except ConnectionError:
            print("No me pude conectar al servidor.")
            return False

        print("Cliente conectado exitosamente al servidor.")
        # Creamos el thread que escucha y guardamos su referencia
        self.thread_escuchar = EscucharQThread(self.socket_cliente)
        self.thread_escuchar.senal_mensaje.connect(self.procesar_mensaje)
        self.thread_escuchar.senal_desconectado.connect(self.senal_desconectado)
        self.thread_escuchar.start()
        return True

    # Aquí procesamos el contenido del mensaje.
    # Para este ejemplo, solo lo mandamos al frontend
    def procesar_mensaje(self, mensaje: str) -> None:
        self.senal_actualizar_etiqueta.emit(mensaje)

    def mandar_comando(self, comando: str) -> None:
        try:
            enviar(self.socket_cliente, comando.encode("utf-8"))
        except (ConnectionError, OSError):
            self.senal_desconectado.emit()

    def cerrar(self) -> None:
        # shutdown despierta al recv que está bloqueado en el otro thread
        # (le hace retornar b""), así el thread termina por sí solo
        try:
            self.socket_cliente.shutdown(socket.SHUT_RDWR)
        except OSError:
            pass  # El socket ya estaba desconectado
        self.socket_cliente.close()
        if self.thread_escuchar is not None:
            self.thread_escuchar.wait()
```

### Conectando todo en `main.py`

`main.py` crea el *back-end* y el *front-end*, y conecta sus señales. La ventana (`frontend/ventana.py`) no tiene nada nuevo de esta semana: emite `senal_mandar_comando` al presionar un botón, y tiene los métodos `actualizar_etiqueta` y `mostrar_desconexion`.

```python
    app = QApplication([])
    # Instanciamos nuestro backend y frontend
    backend = Logica(HOST_DEFECTO, PORT_DEFECTO)
    frontend = MainWindow()

    # Conectamos señales
    backend.senal_actualizar_etiqueta.connect(frontend.actualizar_etiqueta)
    backend.senal_desconectado.connect(frontend.mostrar_desconexion)
    frontend.senal_mandar_comando.connect(backend.mandar_comando)
    # Justo antes de que termine la aplicación, cerramos la conexión
    app.aboutToQuit.connect(backend.cerrar)

    if not backend.conectar_a_servidor():
        sys.exit(1)

    # Mostramos la ventana y ejecutamos PyQt
    frontend.show()
    sys.exit(app.exec())
```

`app.aboutToQuit` es una señal que se emite justo antes de que termine la aplicación. La usamos para cerrar la conexión y esperar al *thread* que escucha, igual que en el ejemplo de [cierre limpio](3-qthread.ipynb#Detener-un-QThread-y-cerrar-limpio).

### ¿Qué pasa si el servidor se cae?

Pruébalo: con un cliente abierto, detén el servidor con `Ctrl+C`. Al cerrarse la conexión, `recv` retorna `b""`, `recibir` lanza `ConnectionError`, y el *thread* que escucha emite `senal_desconectado` y termina. La ventana muestra "Se perdió la conexión con el servidor" y deshabilita sus botones, en vez de quedarse pegada o caerse.

Un truco en el método `cerrar`: llamamos a `shutdown` antes de `close`. Cuando cerramos el cliente, el *thread* que escucha está bloqueado en `recv`. `shutdown` hace que ese `recv` retorne de inmediato, y así el *thread* termina solo.

**Tip:** si reinicias el servidor apenas lo detuviste, puede aparecer `OSError: [Errno 98] Address already in use`. El sistema operativo mantiene el puerto reservado un rato después de cerrar conexiones. Por eso ambos servidores llaman a `setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)` antes del `bind`, lo que permite reutilizar el puerto de inmediato. Ojo: `SO_REUSEADDR` no sirve si **otro programa está escuchando** en ese puerto, por ejemplo el servidor del otro ejemplo, que usa el mismo puerto. En ese caso el error aparece igual, y la solución es cerrar ese otro servidor.

## Ejemplo avanzado (`2_ejemplo_avanzado`)

En este ejemplo hay interacción **entre clientes**, con el servidor como intermediario. Al conectarse, cada cliente le pide al servidor la hora de Chile y la muestra como un reloj. Para simplificar, el servidor usa una diferencia fija con UTC: la de Chile en **horario de verano** (UTC−3). También puede pedir la hora de Japón, o pedir **adelantar o retrasar el reloj en 1 hora**. Estas dos últimas acciones afectan **a todos los clientes conectados**. Te recomendamos probarlo con al menos dos clientes.

Esta vez los mensajes son diccionarios serializados con JSON:

| Quién envía | Mensaje | Respuesta del servidor |
|---|---|---|
| Cliente | `{"comando": "hora", "pais": "chile"}` | `{"tipo": "hora", "hora": "2026-11-02T15:04:05"}`, solo a ese cliente |
| Cliente | `{"comando": "cambio", "accion": "adelantar"}` | `{"tipo": "cambio", "accion": "adelantar"}`, a **todos** los clientes |

### El servidor

El servidor guarda en una lista los *sockets* de todos los clientes conectados, para poder enviarles mensajes a todos. Esa lista es compartida por todos los *threads* que atienden clientes, así que la protegemos con un `Lock`, como vimos en la semana de *threading*. Además, cuando un cliente se desconecta, lo sacamos de la lista en el bloque `finally`.

Este código se encuentra en el archivo `2_ejemplo_avanzado/servidor/main.py`.

```python
class Servidor:
    def __init__(self, host: str, port: int) -> None:
        # La lista de clientes es compartida por todos los threads que
        # atienden clientes, así que la protegemos con un Lock (semana 9)
        self.clientes = []
        self.lock_clientes = Lock()
        # Dos threads pueden enviarle al mismo cliente a la vez (la respuesta
        # a "hora" y un envío a todos). Este lock evita que los bytes de
        # ambos mensajes se mezclen en el socket y rompan el protocolo
        self.lock_envio = Lock()

        # Creamos el socket, hacemos bind y listen
        self.sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
        # Permite reutilizar el puerto apenas se cierra el servidor. Sin esto,
        # al reiniciarlo rápido aparece "OSError: Address already in use"
        self.sock.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)
        self.sock.bind((host, port))
        self.sock.listen()

    def escuchar_conexiones(self) -> None:
        print("Empezando a escuchar clientes....")
        while True:
            # Aceptamos a un cliente
            socket_cliente, direccion = self.sock.accept()
            print("Conexión aceptada desde", direccion)

            # Guardamos el socket de este cliente en la lista
            with self.lock_clientes:
                self.clientes.append(socket_cliente)

            # Creamos un thread encargado de escuchar a ese cliente
            thread = Thread(
                target=self.escuchar_cliente,
                args=(socket_cliente, direccion),
                daemon=True,
            )
            thread.start()

    # Método encargado de escuchar a 1 cliente
    def escuchar_cliente(self, socket_cliente: socket.socket, direccion: tuple) -> None:
        try:
            while True:
                mensaje = json.loads(recibir(socket_cliente).decode("utf-8"))
                print(f"[{direccion}] mandó un mensaje: {mensaje}")
                self.procesar_mensaje(mensaje, socket_cliente)

        except ConnectionError:
            print(f"[{direccion}] se desconectó")
        finally:
            # Sacamos al cliente de la lista, para no enviarle más mensajes
            with self.lock_clientes:
                self.clientes.remove(socket_cliente)
            socket_cliente.close()

    def procesar_mensaje(self, mensaje: dict, socket_cliente: socket.socket) -> None:
        if mensaje["comando"] == "cambio":
            # "adelantar" o "retrasar" afecta a TODOS los clientes conectados
            respuesta = {"tipo": "cambio", "accion": mensaje["accion"]}
            self.enviar_a_todos(respuesta)

        elif mensaje["comando"] == "hora":
            # La hora de un país se responde solo a quien la pidió
            horas = DIFERENCIAS_HORARIAS[mensaje["pais"]]
            ahora_utc = datetime.datetime.now(datetime.UTC)
            hora_pais = ahora_utc + datetime.timedelta(hours=horas)
            # Enviamos la hora como texto (ISO 8601), sin zona horaria
            respuesta = {"tipo": "hora", "hora": hora_pais.replace(tzinfo=None).isoformat()}
            self.enviar(socket_cliente, respuesta)

    def enviar(self, socket_cliente: socket.socket, mensaje: dict) -> None:
        # Cada mensaje se envía completo antes de que empiece el siguiente
        with self.lock_envio:
            enviar(socket_cliente, json.dumps(mensaje).encode("utf-8"))

    def enviar_a_todos(self, mensaje: dict) -> None:
        # Copiamos la lista dentro de lock_clientes, y enviamos fuera de él
        with self.lock_clientes:
            clientes = list(self.clientes)
        for socket_cliente in clientes:
            try:
                self.enviar(socket_cliente, mensaje)
            except OSError:
                # Ese cliente se desconectó: su propio thread lo sacará de la lista
                pass
```

En `enviar_a_todos`, copiamos la lista dentro del `with` y enviamos fuera de él, para no tener el *lock* de la lista tomado mientras esperamos a la red.

Además, `enviar` usa un segundo *lock*, `lock_envio`. Dos *threads* pueden enviarle al mismo cliente a la vez (por ejemplo, la respuesta a "hora" y un envío a todos), y sin el *lock* los *bytes* de ambos mensajes podrían mezclarse en el *socket* y romper el protocolo. Este *lock* sí se mantiene tomado mientras se envía, pero solo hace esperar a otros envíos.

### El cliente: un *thread* para escuchar y un `QTimer` para el reloj

El *back-end* del cliente usa las dos herramientas de esta semana, cada una para lo que corresponde:

- **Recibir mensajes bloquea**: se usa el mismo `EscucharQThread` del ejemplo simple, ahora con mensajes JSON.
- **El reloj avanza 1 segundo cada 1 segundo**: es una tarea periódica y rápida, así que se usa un `QTimer`. Como todo ocurre en el *thread* principal, `self.hora_actual` no necesita ningún *lock*.

Este código se encuentra en el archivo `2_ejemplo_avanzado/cliente/backend/logica.py`.

```python
class Logica(QObject):

    senal_actualizar_etiqueta = pyqtSignal(str)
    senal_desconectado = pyqtSignal()

    def __init__(self, host: str, port: int) -> None:
        super().__init__()
        self.host = host
        self.port = port
        self.socket_cliente = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
        self.thread_escuchar = None

        # El reloj avanza 1 segundo cada 1 segundo: una tarea periódica y
        # corta, así que usamos un QTimer. Todo ocurre en el thread
        # principal, por lo que self.hora_actual no necesita un lock
        self.hora_actual = None
        self.timer_reloj = QTimer(self)
        self.timer_reloj.setInterval(1000)
        self.timer_reloj.timeout.connect(self.avanzar_reloj)

    def conectar_a_servidor(self) -> bool:
        try:
            self.socket_cliente.connect((self.host, self.port))
        except ConnectionError:
            print("No me pude conectar al servidor.")
            return False

        print("Cliente conectado exitosamente al servidor.")
        # Recibir mensajes es bloqueante, así que usamos un QThread
        self.thread_escuchar = EscucharQThread(self.socket_cliente)
        self.thread_escuchar.senal_mensaje.connect(self.procesar_mensaje)
        self.thread_escuchar.senal_desconectado.connect(self.desconectado)
        self.thread_escuchar.start()
        self.pedir_hora("chile")
        return True

    # Aquí procesamos el contenido del mensaje
    def procesar_mensaje(self, mensaje: dict) -> None:
        if mensaje["tipo"] == "hora":
            self.hora_actual = datetime.datetime.fromisoformat(mensaje["hora"])
            if not self.timer_reloj.isActive():
                self.timer_reloj.start()

        elif mensaje["tipo"] == "cambio" and self.hora_actual is not None:
            una_hora = datetime.timedelta(hours=1)
            if mensaje["accion"] == "adelantar":
                self.hora_actual += una_hora
            else:  # mensaje["accion"] == "retrasar"
                self.hora_actual -= una_hora

        self.actualizar_etiqueta()

    def avanzar_reloj(self) -> None:
        self.hora_actual += datetime.timedelta(seconds=1)
        self.actualizar_etiqueta()

    def actualizar_etiqueta(self) -> None:
        if self.hora_actual is not None:
            self.senal_actualizar_etiqueta.emit(self.hora_actual.strftime("%Y-%m-%d %H:%M:%S"))

    def desconectado(self) -> None:
        self.timer_reloj.stop()
        self.senal_desconectado.emit()
```

### Acotación: ¿y con `threading`?

El *thread* que escucha también se puede implementar con `threading.Thread`, igual que en el primer ejemplo del *notebook* de *QThreads*: el *thread* recibe una señal definida en `Logica` y la emite con cada mensaje. El resultado es el mismo, porque la regla de oro se cumple igual: el *thread* nunca toca la interfaz. Gracias a la separación *front-end*/*back-end*, ese cambio solo afectaría a `backend/logica.py`, y la ventana no se enteraría.